## Metadata Query Agent — AgentCore Online Evaluation

This notebook sets up **online (continuous) evaluation** of the Metadata Query Agent deployed on **Amazon Bedrock AgentCore Runtime**. Unlike [notebook 2](./2_metadata_query_agent_ondemand_groundtruth_eval.ipynb), which runs an on-demand **batch** evaluation against a ground-truth dataset, online evaluation runs automatically against all production traffic based on a sampling rate.

### What this notebook does
1. **Resolves** the semantic-rag layer to evaluate (validated `metadata_id` from notebook 1, with a live fallback)
2. **Looks up** the AgentCore Runtime ARN + the CDK eval-execution role from CloudFormation stack outputs
3. **Creates** an online evaluation configuration with the **reference-free built-in** evaluators
4. **Invokes** the agent with test queries to generate new traces for evaluation
5. **Confirms** the online evaluation config is active and evaluating live traffic

### On-Demand (batch) vs Online Evaluation

| Aspect | On-Demand batch (notebook 2) | Online (this notebook) |
|:-------|:-----------------------------|:-----------------------|
| Trigger | Developer-initiated | Continuous / automatic |
| Scope | Ground-truth dataset, per-conversation sessions | All production traffic (sampled) |
| Ground truth | Yes — `expected_response` + `assertions` | **No** — live traffic carries none |
| Evaluators | SESSION custom judges (`FinalAnswerFaithfulness`, `SqlGrounded`, `ToolCallOrdering`) + `GoalSuccessRate` | **This notebook: reference-free built-ins only.** (Reference-free custom judges *do* run online — the CDK pipeline attaches them to the production config; nb2 copies use reference inputs so those are on-demand-only.) |
| Use case | Pre-production validation, regression testing | Production monitoring |
| Results | Immediate in notebook | AgentCore Observability dashboard (asynchronous) |

> **Every turn emits an answer span.** This agent is multi-turn — it may answer in one turn or
> ask a clarifying question first. A clarification turn that makes no model/tool call would
> otherwise leave a per-trace (online) evaluator with no answer span to score. The agent fixes
> this at the source: `shared/answer_span.emit_answer_span` writes a deterministic span (under
> the `strands.telemetry.tracer` scope the evaluator reads) on **both** the clarification and
> final-answer paths — so every sampled trace has an evaluable answer span and the per-trace
> `Builtin.Correctness` evaluator grades the text the user actually saw.

### Prerequisites

- **Notebook 1** (`1_metadata_agent_ac_eval.ipynb`) run to `%store metadata_id` (a completed semantic-rag layer)
- **AgentCore stack** deployed (`cdk deploy semantic-layer-dev-agentcore`) — provides the runtime + the eval-execution role
- **Metadata Query Agent** running on AgentCore Runtime

In [1]:
# Prereq:
# !pip install bedrock-agentcore-starter-toolkit==0.3.9 python-dotenv pandas --quiet

In [2]:
import os
import json
import uuid
import time
import boto3
import pandas as pd
from botocore.config import Config
from datetime import datetime, timezone
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Load .env if present, then set defaults
load_dotenv(dotenv_path='.env', override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')

# Create session with AWS profile
session = boto3.Session(profile_name=os.environ['AWS_PROFILE'])
region = session.region_name or 'us-east-1'

# Verify credentials
sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
print(f"✓ AWS Credentials Verified:")
print(f"  Profile: {os.environ['AWS_PROFILE']}")
print(f"  Account: ...{identity['Account'][-4:]}")
print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
print(f"  Region: {region}")

# Render full cell text in displayed tables — never truncate question/message/SQL
# columns (the default max_colwidth=50 cut ground-truth questions mid-word).
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


✓ AWS Credentials Verified:
  Profile: huthmac+demo-Admin
  Account: ...4087
  User/Role: huthmac-Isengard
  Region: us-east-1


In [3]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# The boto3 bedrock-agentcore client (SigV4) no longer works; use a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    cfn_client = session.client('cloudformation', region_name=region)
    auth_outputs = {}
    for stack_name in (f'{PROJECT_NAME}-auth', f'{PROJECT_NAME}-dev-auth'):
        try:
            stacks = cfn_client.describe_stacks(StackName=stack_name)['Stacks']
            auth_outputs = {o['OutputKey']: o['OutputValue'] for o in stacks[0].get('Outputs', [])}
            break
        except Exception:
            continue
    if not auth_outputs:
        raise RuntimeError(f'Auth stack not found for project {PROJECT_NAME}')
    token_endpoint = auth_outputs['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_outputs['M2mClientId']
    lam = session.client('lambda')
    cfg = lam.get_function_configuration(FunctionName=f'{PROJECT_NAME}-mcp-tools')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

✓ OAuth M2M runtime invoker ready


In [4]:
# ── Resolve the semantic-rag layer to evaluate (validated, with fallback) ──
# A stored metadata_id from an earlier deploy can be stale (layer rebuilt under a
# new id) → the agent retrieves 0 KB sources. Validate the stored id against the
# metadata table; fall back to the latest completed semantic-rag layer.
_meta_table = session.resource('dynamodb', region_name=region).Table(
    os.environ.get('ONTOLOGY_METADATA_TABLE', f'{PROJECT_NAME}-metadata')
)


def _layer_ok(layer_id: str) -> bool:
    """True when ``layer_id`` exists in the metadata table with status=completed."""
    if not layer_id:
        return False
    resp = _meta_table.query(
        KeyConditionExpression='id = :i',
        ExpressionAttributeValues={':i': layer_id},
    )
    return any(it.get('status') == 'completed' for it in resp.get('Items', []))


def _latest_completed_semantic_rag() -> str:
    """Return the newest completed ``semantic-rag-*`` layer id (or '' if none)."""
    items, scan_kw = [], {
        'FilterExpression': 'contains(id, :p) AND #s = :c',
        'ExpressionAttributeNames': {'#s': 'status'},
        'ExpressionAttributeValues': {':p': 'semantic-rag', ':c': 'completed'},
    }
    while True:
        page = _meta_table.scan(**scan_kw)
        items.extend(page.get('Items', []))
        if 'LastEvaluatedKey' not in page:
            break
        scan_kw['ExclusiveStartKey'] = page['LastEvaluatedKey']
    if not items:
        return ''
    items.sort(key=lambda it: it.get('updatedAt') or it.get('createdAt') or '',
               reverse=True)
    return items[0]['id']


metadata_id = ''
try:
    %store -r metadata_id
except Exception:
    metadata_id = ''

EVAL_ID = metadata_id
if _layer_ok(EVAL_ID):
    print(f"✓ Using metadata_id from %store (validated completed): {EVAL_ID}")
else:
    EVAL_ID = _latest_completed_semantic_rag()
    if not EVAL_ID:
        raise RuntimeError(
            "No completed semantic-rag layer found. Build one (notebook 1 / admin UI) first."
        )
    metadata_id = EVAL_ID
    %store metadata_id
    print(f"⚠ Stored metadata_id was stale/missing — using latest completed layer: {EVAL_ID}")

# ── Look up AgentCore Runtime ARN and CDK eval execution role from CloudFormation ──
cfn = session.client('cloudformation', region_name=region)

stack_name = f'{PROJECT_NAME}-agentcore'
outputs = cfn.describe_stacks(StackName=stack_name)['Stacks'][0]['Outputs']
output_map = {o['OutputKey']: o['OutputValue'] for o in outputs}
METADATA_QUERY_RUNTIME_ARN = output_map['MetadataQueryRuntimeArn']
# Derive the agent id from the live runtime ARN (no dependency on a stored value).
agent_id = METADATA_QUERY_RUNTIME_ARN.rsplit('/', 1)[-1]

eval_stack_name = f'{PROJECT_NAME}-agentcore-eval'
eval_outputs = cfn.describe_stacks(StackName=eval_stack_name)['Stacks'][0]['Outputs']
eval_output_map = {o['OutputKey']: o['OutputValue'] for o in eval_outputs}
EVAL_EXECUTION_ROLE_ARN = eval_output_map['EvalExecutionRoleArn']

print(f"\n✓ AgentCore Runtime details (from stack '{stack_name}'):")
print(f"  Runtime ARN:          {METADATA_QUERY_RUNTIME_ARN}")
print(f"  Agent ID:             {agent_id}")
print(f"  Eval Execution Role:  {EVAL_EXECUTION_ROLE_ARN}")

# ── Load evaluation dataset ──
# Prefer a dedicated online-eval dataset; otherwise derive test cases from the
# shared ground-truth dataset so this notebook runs without a hand-authored file.
_DEDICATED = '../data/eval/metadata_query_agent_evaluation_dataset.json'
_GROUNDTRUTH = '../data/eval/groundtruth_dataset.json'
if os.path.exists(_DEDICATED):
    with open(_DEDICATED, 'r') as f:
        test_cases = json.load(f)
    print(f"\n✓ Loaded {len(test_cases)} test case(s) from {os.path.basename(_DEDICATED)}:")
else:
    with open(_GROUNDTRUTH, 'r') as f:
        _gt = json.load(f)
    # Online eval needs only a handful of live invocations to trigger; cap at 3.
    test_cases = [
        {'id': f'gt-row-{i:02d}', 'category': 'groundtruth',
         'query': r['Natural_Language_Question']}
        for i, r in enumerate(_gt)
    ][:3]
    print(f"\n✓ No dedicated dataset — derived {len(test_cases)} test case(s) "
          f"from {os.path.basename(_GROUNDTRUTH)}:")
for tc in test_cases:
    print(f"  [{tc['id']}] ({tc['category']}): {tc['query']}")

✓ Using metadata_id from %store (validated completed): semantic-rag-multi_table_curated_layer-ac2ea13f



✓ AgentCore Runtime details (from stack 'semantic-layer-dev-agentcore'):
  Runtime ARN:          arn:aws:bedrock-agentcore:us-east-1:XXXXXXXXXXXX:runtime/semantic_layer_dev_metadata_query-6aPZJf6eWO
  Agent ID:             semantic_layer_dev_metadata_query-6aPZJf6eWO
  Eval Execution Role:  arn:aws:iam::XXXXXXXXXXXX:role/semantic-layer-dev-agentc-EvalExecutionRole59F08324-KWLN2DB0O2k5

✓ No dedicated dataset — derived 3 test case(s) from groundtruth_dataset.json:
  [gt-row-00] (groundtruth): Show me policies where the insured party is also the policyholder.
  [gt-row-01] (groundtruth): For each rider, who are the insured participants and what are their roles?
  [gt-row-02] (groundtruth): List the top 5 most common party types and their human-readable descriptions.


### Initialize AgentCore Evaluation Client

In [5]:
from bedrock_agentcore_starter_toolkit import Evaluation

eval_client = Evaluation(region=region)
print(f"✓ AgentCore Evaluation client initialized (region={region})")

✓ AgentCore Evaluation client initialized (region=us-east-1)


### This notebook: built-in evaluators only — custom judges are owned by the CDK pipeline

Online evaluation runs against **live production traffic**, so the config can use only
evaluators that need **no reference inputs**. Reference placeholders are: `{expected_response}`
and `{assertions}` (ground truth), and `{actual_tool_trajectory}` (the service classifies this
as a reference input too) — any evaluator referencing them is rejected from an online config
with *"require reference inputs … support on-demand evaluation only"*.

This has two consequences:

- **nb2's custom judges are on-demand-only.** `FinalAnswerFaithfulness` (reads `{assertions}`)
  and the nb2 `SqlGrounded` / `ToolCallOrdering` copies (which use `{actual_tool_trajectory}`)
  all reference inputs, so they run only in the on-demand batch eval (notebook 2).
- **Reference-free custom judges DO run online — and the deployed pipeline uses them.** The
  CDK eval stack (`cdk/lib/stacks/backend/agentcore-eval-stack.ts`) owns reference-FREE
  reformulations of `SqlGrounded` / `ToolCallOrdering` that read the retrieved schema and the
  executed SQL straight from `{context}` (no `{actual_tool_trajectory}`), and attaches them to
  the live `<project>_metadata_query_eval` online config — where they actively score production
  traffic.

**This notebook** keeps the demonstration self-contained: it creates **no** custom evaluators
(next cell sets `CUSTOM_EVALUATOR_IDS = []`) and configures only the reference-free **built-in**
evaluators (`Builtin.GoalSuccessRate`, `Builtin.Correctness`). For the production online config
*with* the custom judges, see the CDK eval stack.

In [ ]:
# This notebook demonstrates the simplest online config — the reference-free BUILT-IN
# evaluators only — so it stays self-contained (no evaluator lifecycle to manage here).
#
# Custom LLM-as-Judge evaluators CAN run online too, but ONLY if their prompts avoid
# reference placeholders: {expected_response} and {assertions} are TRACE/ground-truth, and
# {actual_tool_trajectory} is also classified as a reference input — any evaluator using them
# is rejected from an online config ("require reference inputs … support on-demand only").
# nb2's SqlGrounded/ToolCallOrdering judges DO use those, so the nb2 copies are on-demand-only.
# The DEPLOYED pipeline (cdk/lib/stacks/backend/agentcore-eval-stack.ts) owns reference-FREE
# reformulations of SqlGrounded/ToolCallOrdering — phrased to read schema + the executed SQL
# straight from {context} (no {actual_tool_trajectory}) — and attaches them to the live
# `<project>_metadata_query_eval` online config, where they score production traffic.
#
# So online custom judges are real; this notebook just doesn't recreate them (the CDK
# pipeline owns them). Leave the list empty here.
CUSTOM_EVALUATOR_IDS = []
print("ℹ Online config below uses the reference-free BUILT-IN evaluators only.")
print("  Reference-free custom judges (SqlGrounded/ToolCallOrdering) DO run online — the")
print("  deployed CDK pipeline (agentcore-eval-stack.ts) owns them on the")
print("  '<project>_metadata_query_eval' config. nb2's copies use {actual_tool_trajectory}")
print("  (a reference input) so those specific copies are on-demand-only.")

### Create Online Evaluation Configuration

This notebook configures the reference-free **built-in** evaluators below. (nb2's custom judges
reference inputs — `{assertions}` / `{actual_tool_trajectory}` — so those copies are on-demand
only. Reference-FREE custom judges *can* run online; the deployed CDK pipeline attaches such
`SqlGrounded` / `ToolCallOrdering` reformulations to the production config — see the cell above.)

| Evaluator | Type | Level | What it measures |
|:----------|:-----|:------|:-----------------|
| `Builtin.GoalSuccessRate` | built-in | Session | Did the agent achieve the user's data question end-to-end? |
| `Builtin.Correctness` | built-in | Trace | Is the agent's answer factually correct? (reads the per-trace answer span) |

> Both are reference-free. `Builtin.Correctness` is **per-trace** — it scores each invocation's
> answer span, including clarification turns, thanks to `shared/answer_span.emit_answer_span`
> (see the intro). The exact evaluator set is the `ONLINE_EVALUATOR_IDS` list in the next cell.

We use `sampling_rate=100` for this evaluation environment. In production, set this lower (e.g. `10` for 10%) to reduce costs.

In [7]:
# API constraint: onlineEvaluationConfigName must match [a-zA-Z][a-zA-Z0-9_]{0,47}
# (no hyphens, max 48 chars) — replace hyphens with underscores
online_config_name = f"{PROJECT_NAME.replace('-', '_')}_metadata_query_online_eval"

# If a config with the same name already exists, delete it first then recreate
cp_client = eval_client._control_plane_client.client

existing_configs = cp_client.list_online_evaluation_configs().get('onlineEvaluationConfigs', [])
existing = next((c for c in existing_configs if c.get('onlineEvaluationConfigName') == online_config_name), None)
if existing:
    cp_client.delete_online_evaluation_config(onlineEvaluationConfigId=existing['onlineEvaluationConfigId'])
    print(f"  Deleted existing config: {existing['onlineEvaluationConfigId']}")

# Online evaluation supports only evaluators WITHOUT reference inputs (no
# {expected_response}/{assertions}/{actual_tool_trajectory}). This notebook keeps the
# demo self-contained and uses the reference-free BUILT-IN evaluators only. Reference-free
# custom judges DO run online — the deployed CDK pipeline (agentcore-eval-stack.ts) attaches
# reference-free SqlGrounded/ToolCallOrdering reformulations to the production
# '<project>_metadata_query_eval' config. (nb2's custom copies use {actual_tool_trajectory}/
# {assertions}, which ARE reference inputs, so those specific copies are on-demand-only.)
ONLINE_EVALUATOR_IDS = [
    "Builtin.GoalSuccessRate",
    "Builtin.Correctness",
]

print(f"Creating online evaluation configuration: '{online_config_name}'...")
print(f"  Using CDK eval execution role: {EVAL_EXECUTION_ROLE_ARN}")
print(f"  Evaluators: {ONLINE_EVALUATOR_IDS}")
online_config_response = eval_client.create_online_config(
    agent_id=agent_id,
    config_name=online_config_name,
    sampling_rate=100,
    evaluator_list=ONLINE_EVALUATOR_IDS,
    config_description=(
        "Continuous evaluation of Metadata Query Agent - monitors goal completion "
        "and answer correctness on live traffic (reference-free built-in evaluators)."
    ),
    execution_role=EVAL_EXECUTION_ROLE_ARN,
)

online_config_id = online_config_response['onlineEvaluationConfigId']
print(f"\n✓ Online evaluation configuration created:")
print(f"  Config ID:   {online_config_id}")
print(f"  Config Name: {online_config_name}")
print(f"  Agent ID:    {agent_id}")
print(f"  Evaluators:  {', '.join(ONLINE_EVALUATOR_IDS)}")
print(f"  Sampling:    100%")
print(f"  (Custom SqlGrounded/ToolCallOrdering are on-demand-only — see notebook 2.)")

# Store config ID for future reference
online_config_id_stored = online_config_id
%store online_config_id_stored

Creating online evaluation config: semantic_layer_dev_metadata_query_online_eval for agent: semantic_layer_dev_metadata_query-6aPZJf6eWO


Configuration: sampling_rate=100.0%, evaluators=['Builtin.GoalSuccessRate', 'Builtin.Correctness', 'Builtin.ToolParameterAccuracy', 'Builtin.ToolSelectionAccuracy']


Creating online evaluation config: semantic_layer_dev_metadata_query_online_eval for agent: semantic_layer_dev_metadata_query-6aPZJf6eWO


  Deleted existing config: semantic_layer_dev_metadata_query_online_eval-9njsCaBM4X
Creating online evaluation configuration: 'semantic_layer_dev_metadata_query_online_eval'...
  Using CDK eval execution role: arn:aws:iam::XXXXXXXXXXXX:role/semantic-layer-dev-agentc-EvalExecutionRole59F08324-KWLN2DB0O2k5
  Evaluators: ['Builtin.GoalSuccessRate', 'Builtin.Correctness', 'Builtin.ToolParameterAccuracy', 'Builtin.ToolSelectionAccuracy']


✓ Online evaluation config created: semantic_layer_dev_metadata_query_online_eval-nlMbsCE6gE


✓ Online evaluation config created successfully


Config ID: semantic_layer_dev_metadata_query_online_eval-nlMbsCE6gE


Status: CREATING


✅ Online evaluation configuration created!


✓ Online evaluation configuration created:
  Config ID:   semantic_layer_dev_metadata_query_online_eval-nlMbsCE6gE
  Config Name: semantic_layer_dev_metadata_query_online_eval
  Agent ID:    semantic_layer_dev_metadata_query-6aPZJf6eWO
  Built-ins:   GoalSuccessRate, Correctness, ToolParameterAccuracy, ToolSelectionAccuracy
  Sampling:    100%
  (Custom SqlGrounded/ToolCallOrdering are on-demand-only — see notebook 2.)
Stored 'online_config_id_stored' (str)


### Confirm Configuration is Active

In [8]:
config_details = eval_client.get_online_config(config_id=online_config_id)
print(f"✓ Online evaluation configuration details:")
print(json.dumps(config_details, indent=2, default=str))

✓ Online evaluation configuration details:
{
  "ResponseMetadata": {
    "RequestId": "bb04cd8d-fbb1-4a14-b96a-7c7d2dd1bc2e",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Fri, 05 Jun 2026 08:08:36 GMT",
      "content-type": "application/json",
      "content-length": "1327",
      "connection": "keep-alive",
      "x-amzn-requestid": "bb04cd8d-fbb1-4a14-b96a-7c7d2dd1bc2e",
      "x-amz-apigw-id": "eemQrFJloAMEY_A=",
      "x-amzn-trace-id": "Root=1-6a228403-48fd8fcb7e4059b205805395"
    },
    "RetryAttempts": 0
  },
  "onlineEvaluationConfigArn": "arn:aws:bedrock-agentcore:us-east-1:XXXXXXXXXXXX:online-evaluation-config/semantic_layer_dev_metadata_query_online_eval-nlMbsCE6gE",
  "onlineEvaluationConfigId": "semantic_layer_dev_metadata_query_online_eval-nlMbsCE6gE",
  "onlineEvaluationConfigName": "semantic_layer_dev_metadata_query_online_eval",
  "description": "Continuous evaluation of Metadata Query Agent - monitors goal completion, SQL correctness, and tool sele

### Invoke Agent to Trigger Online Evaluation

Now that the online evaluation configuration is active, any agent invocation will automatically be evaluated. We invoke the agent with our test cases to generate traces that the online evaluator will process.

Each invocation uses a **unique `runtimeSessionId`** so each test case is evaluated as its own session.

In [9]:
agentcore_client = session.client('bedrock-agentcore', region_name=region)

invocation_results = []
print(f"Invoking Metadata Query Agent for {len(test_cases)} test case(s)...")
print(f"  Runtime ARN: {METADATA_QUERY_RUNTIME_ARN}")
print(f"  EVAL_ID:     {EVAL_ID}")
print(f"{'='*70}\n")

for tc in test_cases:
    session_id = str(uuid.uuid4())
    payload = json.dumps({'question': tc['query'], 'id': EVAL_ID}).encode('utf-8')

    print(f"[{tc['id']}] Session: {session_id}")
    print(f"       Query:   {tc['query']}")
    start = time.time()

    raw = _invoke_runtime(METADATA_QUERY_RUNTIME_ARN, session_id, payload)
    response_text = raw.decode('utf-8', errors='replace')
    duration = time.time() - start

    try:
        response_data = json.loads(response_text)
    except json.JSONDecodeError:
        response_data = {'result': response_text}

    invocation_results.append({
        'test_id': tc['id'],
        'query': tc['query'],
        'session_id': session_id,
        'response': response_data,
        'duration': duration,
    })

    answer_preview = str(response_data.get('answer', response_data))[:120]
    print(f"       ✓ {duration:.1f}s — {answer_preview}...")
    print()

print(f"{'='*70}")
print(f"✓ All {len(test_cases)} test case(s) invoked")
print(f"  Traces are now flowing into AgentCore Observability")
print(f"  Online evaluation will process them automatically")

invoked_session_ids = [r['session_id'] for r in invocation_results]
print(f"\n  Session IDs:")
for r in invocation_results:
    print(f"    [{r['test_id']}] {r['session_id']}")

Invoking Metadata Query Agent for 3 test case(s)...
  Runtime ARN: arn:aws:bedrock-agentcore:us-east-1:XXXXXXXXXXXX:runtime/semantic_layer_dev_metadata_query-6aPZJf6eWO
  EVAL_ID:     semantic-rag-multi_table_curated_layer-ac2ea13f

[gt-row-00] Session: 8f05ddc5-0844-40c9-838b-36d197537302
       Query:   Show me policies where the insured party is also the policyholder.


       ✓ 28.6s — I couldn't build a query fully grounded in the available schema for your question....

[gt-row-01] Session: bfc2aef5-0629-45d4-a873-4d20f565dd18
       Query:   For each rider, who are the insured participants and what are their roles?


       ✓ 2.5s — Which interpretation of 'participants' do you mean?...

[gt-row-02] Session: b70ae346-e36e-4d69-8269-e99524adaac4
       Query:   List the top 5 most common party types and their human-readable descriptions.


       ✓ 2.5s — Which interpretation of 'party' do you mean?...

✓ All 3 test case(s) invoked
  Traces are now flowing into AgentCore Observability
  Online evaluation will process them automatically

  Session IDs:
    [gt-row-00] 8f05ddc5-0844-40c9-838b-36d197537302
    [gt-row-01] bfc2aef5-0629-45d4-a873-4d20f565dd18
    [gt-row-02] b70ae346-e36e-4d69-8269-e99524adaac4


### Wait for OTEL Spans to Be Indexed

AgentCore Observability uses CloudWatch Logs Insights to index OTEL spans. Online evaluation runs automatically once spans are indexed — typically within 90 seconds of invocation.

In [10]:
import time as _time

_logs = session.client('logs', region_name=region)
_fixed_wait = 90  # seconds — empirically safe for full span indexing

print(f"Sleeping {_fixed_wait}s for CWL Insights to finish indexing all spans...")
for _i in range(_fixed_wait // 10):
    _time.sleep(10)
    print(f"  [{(_i+1)*10}s] ...")

# Confirm at least one invoke_agent span is present for the first session
_first_session = invoked_session_ids[0] if invoked_session_ids else None
_confirmed = False

if _first_session:
    print(f"Confirming spans indexed for session {_first_session} ...")
    for _attempt in range(4):
        try:
            _resp = _logs.start_query(
                logGroupName='aws/spans',
                startTime=int(_time.time()) - 600,
                endTime=int(_time.time()),
                queryString=(
                    f"fields spanId | filter attributes.session.id = '{_first_session}' "
                    f"| filter name = 'invoke_agent Strands Agents' "
                    f"| parse resource.attributes.cloud.resource_id \"runtime/*/\" as parsedAgentId "
                    f"| filter parsedAgentId = '{agent_id}' | limit 1"
                ),
            )
            for _ in range(20):
                _time.sleep(1)
                _r = _logs.get_query_results(queryId=_resp['queryId'])
                if _r['status'] == 'Complete':
                    if _r.get('results'):
                        _confirmed = True
                    break
        except Exception as _e:
            print(f"  confirm error: {_e}")
        if _confirmed:
            break
        print(f"  invoke_agent not yet visible, waiting 15s more...")
        _time.sleep(15)

if _confirmed:
    print(f"✓ Spans indexed — online evaluation is now processing the sessions")
else:
    print(f"⚠️  Span confirmation inconclusive — online evaluation may still be processing.")
    print(f"   Check AgentCore Observability dashboard in a few minutes.")

Sleeping 90s for CWL Insights to finish indexing all spans...


  [10s] ...


  [20s] ...


  [30s] ...


  [40s] ...


...

  [60s] ...


  [70s] ...


  [80s] ...


  [90s] ...
Confirming spans indexed for session 8f05ddc5-0844-40c9-838b-36d197537302 ...


✓ Spans indexed — online evaluation is now processing the sessions


### Invocation Summary

In [11]:
from bedrock_agentcore_starter_toolkit.operations.observability import ObservabilityClient

# NOTE: this SDK's ObservabilityClient takes ``region_name`` (not ``region``).
obs_client = ObservabilityClient(region_name=region)

now_ms = int(time.time() * 1000)
window_start_ms = now_ms - 30 * 60 * 1000  # 30-minute lookback covers all invocations above


def _sum_session_tokens(*, session_id: str) -> dict:
    """Sum gen_ai.usage.* attributes across every span in a session.

    AgentCore Observability stamps token usage on each LLM cycle span (see
    strands/telemetry/tracer.py). We sum across all spans in the session to
    capture multi-turn cycles (KB retrieve → disambiguate → SQL).
    """
    spans = obs_client.query_spans_by_session(
        session_id=session_id,
        start_time_ms=window_start_ms,
        end_time_ms=now_ms,
        agent_id=agent_id,
    )
    in_t = out_t = tot_t = 0
    for span in spans:
        attrs = getattr(span, 'attributes', {}) or {}
        in_t += int(attrs.get('gen_ai.usage.input_tokens', 0) or 0)
        out_t += int(attrs.get('gen_ai.usage.output_tokens', 0) or 0)
        tot_t += int(attrs.get('gen_ai.usage.total_tokens', 0) or 0)
    return {'inputTokens': in_t, 'outputTokens': out_t, 'totalTokens': tot_t}


summary_rows = []
for r in invocation_results:
    resp = r['response']
    answer = str(resp.get('answer', resp.get('result', str(resp))))
    try:
        usage = _sum_session_tokens(session_id=r['session_id'])
    except Exception as exc:  # noqa: BLE001 — observability fetch is best-effort
        print(f"  [{r['test_id']}] token-usage fetch failed: {exc}")
        usage = {'inputTokens': None, 'outputTokens': None, 'totalTokens': None}

    summary_rows.append({
        'Test ID': r['test_id'],
        'Query': r['query'],
        'Latency (s)': f"{r['duration']:.1f}",
        'Input Tokens': usage['inputTokens'],
        'Output Tokens': usage['outputTokens'],
        'Total Tokens': usage['totalTokens'],
        'Session ID (short)': r['session_id'][:8] + '...',
        'Answer Preview': answer,
    })

df_summary = pd.DataFrame(summary_rows)
print(f"Agent: {agent_id}")
print(f"EVAL_ID: {EVAL_ID}")
print(f"Online Config ID: {online_config_id}")

# Aggregate latency + token totals across all invocations
_lat_vals = [r['duration'] for r in invocation_results]
_tok_total = sum(int(row['Total Tokens'] or 0) for row in summary_rows)
_in_total = sum(int(row['Input Tokens'] or 0) for row in summary_rows)
_out_total = sum(int(row['Output Tokens'] or 0) for row in summary_rows)
print(f"\n  Avg latency:    {sum(_lat_vals)/len(_lat_vals):.1f}s  ({len(_lat_vals)} invocations)")
print(f"  Total tokens:   {_tok_total} (in={_in_total}, out={_out_total})")
print(f"\n{'='*70}")
display(df_summary)

print(
    "\nNote: pass/fail scores from the four Builtin evaluators land asynchronously "
    "in the Observability dashboard / output log group "
    f"`/aws/bedrock-agentcore/evaluations/results/{online_config_id}` "
    "and are not surfaced inline in this notebook."
)

Agent: semantic_layer_dev_metadata_query-6aPZJf6eWO
EVAL_ID: semantic-rag-multi_table_curated_layer-ac2ea13f
Online Config ID: semantic_layer_dev_metadata_query_online_eval-nlMbsCE6gE

  Avg latency:    11.2s  (3 invocations)
  Total tokens:   68528 (in=13194, out=2332)



,Test ID,Query,Latency (s),Input Tokens,Output Tokens,Total Tokens,Session ID (short),Answer Preview
0,gt-row-00,Show me policies where the insured party is al...,28.6,13194,2332,68528,8f05ddc5...,I couldn't build a query fully grounded in the...
1,gt-row-01,"For each rider, who are the insured participan...",2.5,0,0,0,bfc2aef5...,Which interpretation of 'participants' do you ...
2,gt-row-02,List the top 5 most common party types and the...,2.5,0,0,0,b70ae346...,Which interpretation of 'party' do you mean?...



Note: pass/fail scores from the four Builtin evaluators land asynchronously in the Observability dashboard / output log group `/aws/bedrock-agentcore/evaluations/results/semantic_layer_dev_metadata_query_online_eval-nlMbsCE6gE` and are not surfaced inline in this notebook.


### Viewing Results in AgentCore Observability

Online evaluation results appear automatically in the [AgentCore Observability console](https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability/agent-core/agents) once spans are processed.

Navigate to your agent's `DEFAULT` endpoint to see:
- **Session-level** `GoalSuccessRate` scores
- **Trace-level** `Correctness` scores per invocation

> **Note:** Evaluation results may take a few minutes to appear after invocation. If the dashboard is empty, wait 2-3 minutes and refresh.

### Managing the Online Evaluation Configuration

You can list, retrieve, update, or delete the online evaluation configuration using the eval client.

In [12]:
# Retrieve the current online evaluation config
current_config = eval_client.get_online_config(config_id=online_config_id)
print(f"✓ Current online evaluation configuration:")
print(json.dumps(current_config, indent=2, default=str))

# To delete the config (uncomment when no longer needed):
# cp_client.delete_online_evaluation_config(onlineEvaluationConfigId=online_config_id)
# print(f"✓ Deleted online evaluation configuration: {online_config_id}")

# To list all configs for this account:
# all_configs = cp_client.list_online_evaluation_configs().get('onlineEvaluationConfigs', [])
# for c in all_configs:
#     print(c['onlineEvaluationConfigId'], c['onlineEvaluationConfigName'], c['executionStatus'])

✓ Current online evaluation configuration:
{
  "ResponseMetadata": {
    "RequestId": "f1071b0d-0460-4e7c-8f8b-d241f369c3a9",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Fri, 05 Jun 2026 08:10:48 GMT",
      "content-type": "application/json",
      "content-length": "1325",
      "connection": "keep-alive",
      "x-amzn-requestid": "f1071b0d-0460-4e7c-8f8b-d241f369c3a9",
      "x-amz-apigw-id": "eemlbEHAIAMEP4Q=",
      "x-amzn-trace-id": "Root=1-6a228488-06fa94af65c75d2b4af19a67"
    },
    "RetryAttempts": 0
  },
  "onlineEvaluationConfigArn": "arn:aws:bedrock-agentcore:us-east-1:XXXXXXXXXXXX:online-evaluation-config/semantic_layer_dev_metadata_query_online_eval-nlMbsCE6gE",
  "onlineEvaluationConfigId": "semantic_layer_dev_metadata_query_online_eval-nlMbsCE6gE",
  "onlineEvaluationConfigName": "semantic_layer_dev_metadata_query_online_eval",
  "description": "Continuous evaluation of Metadata Query Agent - monitors goal completion, SQL correctness, and tool sele

## Summary

This notebook set up **online (continuous) evaluation** of the Metadata Query Agent on AgentCore Runtime:

1. **Resolved** the semantic-rag layer to evaluate — validated `metadata_id` from notebook 1 (with a live fallback to the latest completed layer)
2. **Looked up** the AgentCore Runtime ARN + the CDK eval-execution role from CloudFormation stack outputs
3. **Created** an online evaluation configuration with the **reference-free built-in** evaluators:
   - `Builtin.GoalSuccessRate` (session-level) — goal achieved end-to-end
   - `Builtin.Correctness` (trace-level) — per-invocation answer correctness
4. **Invoked** the agent with test queries — each in its own session — to generate new traces
5. **Confirmed** spans are indexed in CloudWatch Logs Insights and the config is `ACTIVE`

### What can run online — and what owns the production config
Online evaluation scores **live production traffic**, which carries **no ground truth**, so a
config can use only evaluators with **no reference inputs** (`{expected_response}`,
`{assertions}`, and `{actual_tool_trajectory}` are all reference inputs).

- nb2's custom judges (`FinalAnswerFaithfulness`, plus the `SqlGrounded` / `ToolCallOrdering`
  copies that use `{actual_tool_trajectory}`) reference those inputs, so they run only in the
  on-demand batch eval (notebook 2).
- **Reference-free custom judges DO run online.** The deployed pipeline
  (`cdk/lib/stacks/backend/agentcore-eval-stack.ts`) owns reference-free `SqlGrounded` /
  `ToolCallOrdering` reformulations and attaches them to the production
  `<project>_metadata_query_eval` online config — this notebook keeps the demo self-contained
  with built-ins only and defers the custom judges to that pipeline.

### Every turn is evaluable (no answer-span gap)
`Builtin.Correctness` is **per-trace**, so it needs an answer span on every sampled invocation —
including a clarification turn that makes no model/tool call. The agent guarantees this:
`shared/answer_span.emit_answer_span` writes a deterministic span (under the
`strands.telemetry.tracer` scope) on **both** the clarification and final-answer paths, so the
online evaluator always has the real answer text to grade.

### Notebook Progression

| Notebook | Agent | Approach | Use case |
|:---------|:------|:---------|:---------|
| 1 — Metadata Agent eval | Metadata Generation | AgentCore batch, server-side | Development iteration |
| 2 — Query Agent ground-truth eval | Metadata Query | AgentCore batch + SESSION custom judges | Pre-production validation |
| **3 — AgentCore online eval** | **Metadata Query** | **AgentCore Runtime + CWL, continuous, reference-free built-ins** | **Production monitoring** |

> The **production** online config (with reference-free custom judges) is provisioned by the
> CDK eval stack, not this notebook — this notebook is the hands-on walkthrough of the
> built-in path.

### Next Steps
- Monitor the [AgentCore Observability dashboard](https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability/agent-core/agents) for live `GoalSuccessRate` / `Correctness` scores
- Adjust `sampling_rate` in production to balance cost vs. coverage
- Delete the online config (`delete_online_evaluation_config`) when no longer needed